# Treinamento e reproducao dos modelos

Este notebook registra o fluxo usado para preparar o dataset e treinar as duas CNNs do projeto: `BaselineCNN` e `ImprovedCNN`. As arquiteturas sao proprias, implementadas em PyTorch, sem modelos pre-treinados e sem transfer learning.

Para reproduzir o treinamento completo, mantenha o dataset EuroSAT RGB completo em `EuroSAT_RGB/` na raiz do projeto. A amostra versionada no repositorio e suficiente para a demonstracao, mas nao para repetir as metricas finais de treinamento.

In [ ]:
from pathlib import Path

project_root = Path.cwd()
required_paths = [
    project_root / "src/models/baseline_cnn.py",
    project_root / "src/models/improved_cnn.py",
    project_root / "scripts/train_models.py",
    project_root / "EuroSAT_RGB",
]

for path in required_paths:
    print(f"{path}: {'OK' if path.exists() else 'NAO ENCONTRADO'}")

## 1. Preparacao dos dados

A etapa abaixo gera o manifesto, os splits estratificados de treino/validacao/teste e as estatisticas de normalizacao calculadas somente no treino.

In [ ]:
!python -m src.data.prepare_dataset

## 2. Treinamento das CNNs

O comando abaixo reproduz a configuracao registrada no relatorio: `BaselineCNN` por 8 epocas e `ImprovedCNN` por 12 epocas, usando `AdamW`, `CrossEntropyLoss`, batch size 256 e seed 42.

A execucao completa pode demorar, principalmente em CPU.

In [ ]:
!python scripts/train_models.py --batch-size 256 --epochs-baseline 8 --epochs-improved 12 --force-cpu

## 3. Conferencia das metricas registradas

Depois do treinamento, as metricas finais ficam em `models/metrics.json`, os pesos em `models/*.pt` e os graficos em `reports/`.

In [ ]:
import json
from pathlib import Path

metrics = json.loads(Path("models/metrics.json").read_text(encoding="utf-8"))
best = metrics["best_model"]
baseline = metrics["models"]["baseline"]["test"]
improved = metrics["models"]["improved"]["test"]

print(f"Melhor modelo: {best['display_name']}")
print(f"BaselineCNN test accuracy: {baseline['accuracy']:.4%}")
print(f"ImprovedCNN test accuracy: {improved['accuracy']:.4%}")
print(f"Meta de 88% atingida: {best['target_reached']}")